In [17]:
import pandas as pd
import heapq
import time
from collections import defaultdict
#from google.colab import drive
!pip install openpyxl
# --- quick logger setup (add once before using `logger`) ---
import logging, sys
logger = logging.getLogger("mfas")
logger.setLevel(logging.INFO)
if not logger.handlers:
    h = logging.StreamHandler(sys.stdout)
    h.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(name)s | %(message)s",
                                     datefmt="%H:%M:%S"))
    logger.addHandler(h)
    logger.propagate = False


# === Mount Google Drive ===
#drive.mount('/content/drive', force_remount=True)

def read_graph(csv_path):
    df = pd.read_csv(csv_path)
    df.rename(columns={
        'Source Node  ID': 'source',
        'Target Node ID': 'target',
        'Edge Weight': 'weight'
    }, inplace=True)
    df['source'] = df['source'].astype(str)
    df['target'] = df['target'].astype(str)
    edges = list(df.itertuples(index=False, name=None))
    node_set = sorted(set(u for u, v, _ in edges).union(v for u, v, _ in edges))
    node_to_index = {node: i for i, node in enumerate(node_set)}
    index_to_node = {i: node for node, i in node_to_index.items()}
    edges_indexed = [(node_to_index[u], node_to_index[v], float(w)) for (u, v, w) in edges]
    return edges_indexed, node_to_index, index_to_node

In [18]:
def load_initial_scores(csv_path, node_to_index):
    df = pd.read_csv(csv_path)
    df['Node ID'] = df['Node ID'].astype(str).str.strip()
    rank_map = {row['Node ID']: row['Order'] for _, row in df.iterrows()}

    scores = {}
    for node_str, idx in node_to_index.items():
        if node_str in rank_map:
            scores[idx] = int(rank_map[node_str])

    # Assign unique ranks to unranked nodes
    max_rank = max(scores.values(), default=0) + 1
    for node_str, idx in node_to_index.items():
        if idx not in scores:
            scores[idx] = max_rank
            max_rank += 1

    return scores


def compute_forward_weight(edges, scores):
    return sum(w for u, v, w in edges if scores[u] < scores[v])
def total_weight(edges):
    return sum(w for _, _, w in edges)

def print_weights(label, edges, scores):
    fw = compute_forward_weight(edges, scores)
    tw = total_weight(edges)
    print(f"[{label}] forward_weight = {fw:.6g} | total_weight = {tw:.6g}")
    return fw, tw


In [24]:
# ====================== PRINT-ONLY WITH CYCLE-CORE SAMPLING ======================
import time
import numpy as np
import pandas as pd
from collections import defaultdict, deque

# --- internal deps (safe; do not redefine your helpers) ---
def _build_out_adj(edges_indexed, n):
    out_adj = [[] for _ in range(n)]
    for u, v, w in edges_indexed:
        out_adj[u].append((v, float(w)))
    return out_adj

def _glynn_sample_and_grad(out_adj, n, rng):
    # Rademacher vector
    eps = rng.choice(np.array([-1.0, 1.0], dtype=np.float64), size=n, replace=True)
    eps_prod = float(np.prod(eps))

    # Row sums s_i = Σ_j ε_j A_{ij}
    s = np.zeros(n, dtype=np.float64)
    for i in range(n):
        acc = 0.0
        for j, w in out_adj[i]:
            acc += eps[j] * w
        s[i] = acc

    # Product and leave-one-outs
    zero_idx = [i for i, v in enumerate(s) if v == 0.0]
    if len(zero_idx) == 0:
        P = 1.0
        for v in s: P *= v
        L = [P / s[i] for i in range(n)]
    elif len(zero_idx) == 1:
        z = zero_idx[0]
        P_excl = 1.0
        for i, v in enumerate(s):
            if i != z: P_excl *= v
        P, L = 0.0, [0.0]*n
        L[z] = P_excl
    else:
        P, L = 0.0, [0.0]*n

    scale = 2.0 ** (1 - n)
    perm_sample = scale * eps_prod * P
    coeff = scale * eps_prod

    grad_sample = {}
    for p in range(n):
        Lp = L[p]
        if Lp == 0.0:
            continue
        row_coeff = coeff * Lp
        for q, _w in out_adj[p]:
            grad_sample[(p, q)] = grad_sample.get((p, q), 0.0) + row_coeff * eps[q]
    return perm_sample, grad_sample

def permanent_edge_scores_from_edges_print(
    edges_indexed,
    n,
    samples=2048,
    seed=42,
    print_every=0.02,     # float fraction or int step
    max_seconds=None
):
    """
    Rich-print Monte-Carlo estimator of perm(A) and its per-edge gradients.
    Prints progress % with elapsed, ETA, samples/sec, running mean ± SE, and gradient L1/sample.
    """
    out_adj = _build_out_adj(edges_indexed, n)
    rng = np.random.default_rng(seed)

    perm_sum = 0.0
    perm_sumsq = 0.0
    grad_accum = defaultdict(float)
    grad_l1_cum = 0.0

    step = max(1, int(samples * print_every)) if isinstance(print_every, float) else max(1, int(print_every))
    t0 = time.time()
    actual = 0

    print(f"[perm-MC] seed={seed}, samples={samples}, print_every={step} iters, max_seconds={max_seconds}", flush=True)

    for t in range(1, samples + 1):
        s_val, g = _glynn_sample_and_grad(out_adj, n, rng)
        perm_sum += s_val
        perm_sumsq += (s_val * s_val)
        l1_this = 0.0
        for k, v in g.items():
            grad_accum[k] += v
            l1_this += abs(v)
        grad_l1_cum += l1_this
        actual = t

        if (t % step == 0) or (t == samples):
            elapsed = time.time() - t0
            rate = t / elapsed if elapsed > 0 else float('inf')
            mu = perm_sum / t
            var = max(0.0, perm_sumsq / t - mu * mu)
            sd = var ** 0.5
            se = (sd / (t ** 0.5)) if t > 0 else float('nan')
            eta = (samples - t) / rate if rate > 0 else float('inf')
            print(
                f"[perm-MC] {t}/{samples} ({100*t/samples:.1f}%) | "
                f"elapsed={elapsed:.1f}s | eta={eta:.1f}s | rate={rate:.1f} samp/s | "
                f"perm≈{mu:.6e} ±{se:.2e} (SE) | grad_L1/sample≈{grad_l1_cum/t:.3e}",
                flush=True
            )

        if max_seconds is not None and (time.time() - t0) >= max_seconds:
            print(f"[perm-MC] ⏱️ time cap hit at t={t}; stopping early.", flush=True)
            break

    inv = 1.0 / float(actual if actual > 0 else 1)
    perm_est = perm_sum * inv
    for k in list(grad_accum.keys()):
        grad_accum[k] *= inv
    print(f"[perm-MC] done: used_samples={actual}, perm_est≈{perm_est:.6e}", flush=True)
    return perm_est, dict(grad_accum)

# ---- cycle-core extraction (peel zero-in/zero-out nodes iteratively) ----
def build_degree_views(edges_indexed, n):
    out_adj = [[] for _ in range(n)]
    in_adj  = [[] for _ in range(n)]
    out_deg = [0]*n
    in_deg  = [0]*n
    for u, v, w in edges_indexed:
        out_adj[u].append((v, w))
        in_adj[v].append((u, w))
        out_deg[u] += 1
        in_deg[v]  += 1
    return out_adj, in_adj, out_deg, in_deg

def cycle_core_subgraph(edges_indexed, n, verbose=True):
    """
    Iteratively remove nodes with zero in-degree or zero out-degree.
    Returns:
      core_edges (renumbered), old2new (dict old->new), keep_nodes (list of kept old IDs)
    """
    out_adj, in_adj, out_deg, in_deg = build_degree_views(edges_indexed, n)
    alive = [True]*n
    q = deque([i for i in range(n) if out_deg[i] == 0 or in_deg[i] == 0])
    removed = 0
    while q:
        i = q.popleft()
        if not alive[i]:
            continue
        alive[i] = False
        removed += 1
        for v, _ in out_adj[i]:
            in_deg[v] -= 1
            if alive[v] and (in_deg[v] == 0 or out_deg[v] == 0):
                q.append(v)
        for u, _ in in_adj[i]:
            out_deg[u] -= 1
            if alive[u] and (in_deg[u] == 0 or out_deg[u] == 0):
                q.append(u)

    keep_nodes = [i for i,a in enumerate(alive) if a]
    old2new = {old:new for new, old in enumerate(keep_nodes)}
    core_edges = []
    for u, v, w in edges_indexed:
        if alive[u] and alive[v]:
            core_edges.append((old2new[u], old2new[v], w))

    if verbose:
        print(f"[core] kept_nodes={len(keep_nodes):,}/{n:,} "
              f"({(len(keep_nodes)/max(1,n))*100:.2f}%) | "
              f"kept_edges={len(core_edges):,}/{len(edges_indexed):,}", flush=True)
    return core_edges, old2new, keep_nodes

# ---- VERBOSE wrapper: uses cycle-core for sampling, prints rich stats ----
def run_perm_scoring_with_metrics_print(
    csv_path,
    initial_ranking_path,
    output_csv,
    samples=512,
    seed=42,
    normalize=True,
    prune_top_k=0,
    prune_only_backward=True,
    print_every=0.05,
    max_seconds=None,
    topk_print=10,               # how many top edges to show (overall + backward)
    show_head=5                  # preview rows written
):
    t_global = time.time()
    print("="*80, flush=True)
    print(f"[stage] START permanent-based scoring", flush=True)
    print(f"[cfg] csv='{csv_path}' | init='{initial_ranking_path}' | out='{output_csv}'", flush=True)
    print(f"[cfg] samples={samples} seed={seed} prune_top_k={prune_top_k} prune_only_backward={prune_only_backward}", flush=True)

    # 1) Load graph + initial ranks (uses your helpers)
    print(f"[stage] Loading graph…", flush=True)
    edges_indexed, node_to_index, index_to_node = read_graph(csv_path)
    n = len(node_to_index)
    m = len(edges_indexed)
    density = (m / (n * (n - 1))) if n > 1 else 0.0
    weights = np.array([w for _, _, w in edges_indexed], dtype=np.float64)
    print(
        f"[info] nodes={n:,} | edges={m:,} | density≈{density:.6f} | "
        f"weight_sum={weights.sum():.6g} | min/mean/p50/p90/p99/max="
        f"{weights.min():.4g}/{weights.mean():.4g}/"
        f"{np.percentile(weights,50):.4g}/{np.percentile(weights,90):.4g}/"
        f"{np.percentile(weights,99):.4g}/{weights.max():.4g}",
        flush=True
    )

    print(f"[stage] Loading initial ranking…", flush=True)
    scores = load_initial_scores(initial_ranking_path, node_to_index)
    unique_ranks = len(set(scores.values()))
    print(f"[info] ranking_coverage={unique_ranks}/{n} unique ranks.", flush=True)

    # BEGIN metrics
    print("[stage] BEGIN metrics …", flush=True)
    fw_begin, tw = print_weights("BEGIN", edges_indexed, scores)

    # BEGIN forward/backward breakdown
    f_cnt = b_cnt = 0
    f_w = b_w = 0.0
    for (u, v, w) in edges_indexed:
        if scores[u] < scores[v]:
            f_cnt += 1; f_w += w
        elif scores[u] > scores[v]:
            b_cnt += 1; b_w += w
    print(f"[info] BEGIN breakdown: forward_edges={f_cnt:,} (w={f_w:.6g}) | backward_edges={b_cnt:,} (w={b_w:.6g})", flush=True)

    # 2) Build cycle core and sample ONLY there
    print(f"[stage] Building cycle core …", flush=True)
    core_edges, old2new, keep_nodes = cycle_core_subgraph(edges_indexed, n, verbose=True)

    if len(keep_nodes) == 0:
        print("[warn] Cycle core is empty → perm(A)=0 on full graph; gradients are 0.", flush=True)
        perm_est = 0.0
        grad_full = {}  # all edges zero
    else:
        n_core = len(keep_nodes)
        print(f"[stage] Sampling on core (n_core={n_core:,}, |E_core|={len(core_edges):,}) …", flush=True)
        t1 = time.time()
        perm_core, grad_core = permanent_edge_scores_from_edges_print(
            core_edges, n_core, samples=samples, seed=seed, print_every=print_every, max_seconds=max_seconds
        )
        print(f"[done] core_sampling_time={time.time()-t1:.2f}s | permanent_core≈{perm_core:.6e}", flush=True)

        # Map gradients back to full (old index space)
        grad_full = {}
        for (u_old, v_old, w) in edges_indexed:
            if (u_old in old2new) and (v_old in old2new):
                grad_full[(u_old, v_old)] = grad_core.get((old2new[u_old], old2new[v_old]), 0.0)
            else:
                grad_full[(u_old, v_old)] = 0.0
        perm_est = perm_core

    # 3) Build DataFrame with scores
    rows = []
    for (u, v, w) in edges_indexed:
        g = float(grad_full.get((u, v), 0.0))
        rows.append((index_to_node[u], index_to_node[v], w, g, u, v))
    df = pd.DataFrame(rows, columns=["source_id","target_id","weight","grad","_u","_v"])
    df["score"] = df["grad"].clip(lower=0.0)
    if normalize:
        ssum = df["score"].sum()
        if ssum > 0:
            df["score"] = df["score"] / ssum
        print(f"[info] score_normalization: sum(scores)={df['score'].sum():.6f}", flush=True)

    # forward/backward flags using your ranks
    df["rank_u"] = df["_u"].map(scores)
    df["rank_v"] = df["_v"].map(scores)
    df["is_backward"] = df["rank_u"] > df["rank_v"]
    df["is_forward"]  = df["rank_u"] < df["rank_v"]

    # Show top-K edges overall
    print(f"[stage] Top-{topk_print} edges by score (overall):", flush=True)
    df_sorted = df.sort_values("score", ascending=False)
    for idx, r in df_sorted.head(topk_print).iterrows():
        print(f"  #{idx}: {r['source_id']}→{r['target_id']}  "
              f"score={r['score']:.3e} grad={r['grad']:.3e} w={r['weight']:.3g}  "
              f"ranks=({int(r['rank_u'])},{int(r['rank_v'])})  Δrank={int(r['rank_u'])-int(r['rank_v'])}",
              flush=True)

    # Show top-K BACKWARD edges
    topb = df[df["is_backward"]].sort_values("score", ascending=False).head(topk_print)
    print(f"[stage] Top-{topk_print} BACKWARD edges by score:", flush=True)
    for idx, r in topb.iterrows():
        print(f"  #{idx}: {r['source_id']}→{r['target_id']}  "
              f"score={r['score']:.3e} grad={r['grad']:.3e} w={r['weight']:.3g}  "
              f"ranks=({int(r['rank_u'])},{int(r['rank_v'])})  Δrank={int(r['rank_u'])-int(r['rank_v'])}",
              flush=True)

    # 4) Choose removals for END state
    df["_rm"] = False
    if prune_top_k and prune_top_k > 0:
        cand = df[df["is_backward"]] if prune_only_backward else df
        drop = cand.sort_values("score", ascending=False).head(int(prune_top_k))
        df.loc[drop.index, "_rm"] = True
        print(f"[stage] Marked {len(drop)} edges for removal (prune_only_backward={prune_only_backward}). Preview:", flush=True)
        for _, r in drop.head(min(topk_print, len(drop))).iterrows():
            print(f"   remove: {r['source_id']}→{r['target_id']}  "
                  f"score={r['score']:.3e} grad={r['grad']:.3e} w={r['weight']:.3g}  "
                  f"ranks=({int(r['rank_u'])},{int(r['rank_v'])})", flush=True)
    else:
        print("[stage] No pruning requested (prune_top_k=0).", flush=True)

    # END weights (after removals)
    remaining_edges = [(int(u), int(v), float(w))
                       for (u, v, w) in df.loc[~df["_rm"], ["_u","_v","weight"]].itertuples(index=False, name=None)]
    print("[stage] END metrics …", flush=True)
    fw_end, _ = print_weights("END", remaining_edges, scores)

    # END breakdown
    f_cnt_end = b_cnt_end = 0
    f_w_end = b_w_end = 0.0
    for (u, v, w) in remaining_edges:
        if scores[u] < scores[v]:
            f_cnt_end += 1; f_w_end += w
        elif scores[u] > scores[v]:
            b_cnt_end += 1; b_w_end += w
    print(f"[info] END breakdown: forward_edges={f_cnt_end:,} (w={f_w_end:.6g}) | backward_edges={b_cnt_end:,} (w={b_w_end:.6g})", flush=True)

    # 5) Write output CSV
    print(f"[stage] Writing CSV → {output_csv}", flush=True)
    df_out = df[["source_id","target_id","weight","grad","score","is_backward","is_forward"]].copy()
    df_out["total_weight_all_edges"] = float(tw)
    df_out["forward_weight_begin"]   = float(fw_begin)
    df_out["forward_weight_end"]     = float(fw_end)
    df_out["permanent_estimate"]     = float(perm_est)
    df_out["removed_edges_count"]    = int(df["_rm"].sum())
    df_out["removed_weight"]         = float(df.loc[df["_rm"], "weight"].sum())
    df_out.sort_values("score", ascending=False, inplace=True, ignore_index=True)
    df_out.to_csv(output_csv, index=False)
    print(f"[done] wrote_rows={len(df_out)}  file='{output_csv}'", flush=True)

    # optional head preview
    if show_head and len(df_out) > 0:
        print(f"[preview] first {show_head} rows:", flush=True)
        print(df_out.head(show_head).to_string(index=False), flush=True)

    total_time = time.time() - t_global
    print(f"[stage] FINISHED in {total_time:.2f}s", flush=True)
    print("="*80, flush=True)

    return {
        "perm_est": float(perm_est),
        "forward_begin": float(fw_begin),
        "forward_end": float(fw_end),
        "total_weight": float(tw),
        "removed_edges": int(df["_rm"].sum()),
        "removed_weight": float(df.loc[df["_rm"], "weight"].sum()),
        "output_csv": output_csv,
    }
# ==================== END PRINT-ONLY WITH CYCLE-CORE SAMPLING ====================


In [25]:
# Baseline (no pruning)
summary0 = run_perm_scoring_with_metrics_print(
    csv_path=csv_path,
    initial_ranking_path=initial_ranking_path,
    output_csv=output_csv.replace(".csv","_p0.csv"),
    samples=256,
    seed=42,
    normalize=True,
    prune_top_k=0,
    prune_only_backward=True,
    print_every=0.05,
)

# Pruned variant (e.g., top-50 backward edges removed before END metric)
summary2 = run_perm_scoring_with_metrics_print(
    csv_path=csv_path,
    initial_ranking_path=initial_ranking_path,
    output_csv=output_csv.replace(".csv","_p50.csv"),
    samples=256,
    seed=42,
    normalize=True,
    prune_top_k=50,
    prune_only_backward=True,
    print_every=0.05,
)

# Quick comparison print
print("\n=== COMPARISON ===")
for k in ["perm_est","forward_begin","forward_end","total_weight","removed_edges","removed_weight"]:
    v0 = summary0[k]
    v2 = summary2[k]
    print(f"{k:>16}: baseline={v0:.6g}   prune={v2:.6g}   Δ={v2 - v0:+.6g}")


[stage] START permanent-based scoring
[cfg] csv='../datasets/connectome_graph.csv' | init='../datasets/35462925.csv' | out='../datasets/connectome_ranking_permenant_p0.csv'
[cfg] samples=256 seed=42 prune_top_k=0 prune_only_backward=True
[stage] Loading graph…
[info] nodes=136,648 | edges=5,657,719 | density≈0.000303 | weight_sum=4.19121e+07 | min/mean/p50/p90/p99/max=2/7.408/4/15/54/2405
[stage] Loading initial ranking…
[info] ranking_coverage=136648/136648 unique ranks.
[stage] BEGIN metrics …
[BEGIN] forward_weight = 3.54629e+07 | total_weight = 4.19121e+07
[info] BEGIN breakdown: forward_edges=4,500,907 (w=3.54629e+07) | backward_edges=1,156,812 (w=6.44922e+06)
[stage] Building cycle core …
[core] kept_nodes=127,156/136,648 (93.05%) | kept_edges=5,607,064/5,657,719
[stage] Sampling on core (n_core=127,156, |E_core|=5,607,064) …
[perm-MC] seed=42, samples=256, print_every=12 iters, max_seconds=None
[perm-MC] 12/256 (4.7%) | elapsed=20.2s | eta=410.6s | rate=0.6 samp/s | perm≈0.00000

[stage] FINISHED in 504.09s
[stage] START permanent-based scoring
[cfg] csv='../datasets/connectome_graph.csv' | init='../datasets/35462925.csv' | out='../datasets/connectome_ranking_permenant_p50.csv'
[cfg] samples=256 seed=42 prune_top_k=50 prune_only_backward=True
[stage] Loading graph…
[info] nodes=136,648 | edges=5,657,719 | density≈0.000303 | weight_sum=4.19121e+07 | min/mean/p50/p90/p99/max=2/7.408/4/15/54/2405
[stage] Loading initial ranking…
[info] ranking_coverage=136648/136648 unique ranks.
[stage] BEGIN metrics …
[BEGIN] forward_weight = 3.54629e+07 | total_weight = 4.19121e+07
[info] BEGIN breakdown: forward_edges=4,500,907 (w=3.54629e+07) | backward_edges=1,156,812 (w=6.44922e+06)
[stage] Building cycle core …
[core] kept_nodes=127,156/136,648 (93.05%) | kept_edges=5,607,064/5,657,719
[stage] Sampling on core (n_core=127,156, |E_core|=5,607,064) …
[perm-MC] seed=42, samples=256, print_every=12 iters, max_seconds=None
[perm-MC] 12/256 (4.7%) | elapsed=20.2s | eta=410.1s | 

[preview] first 5 rows:
         source_id          target_id  weight  grad  score  is_backward  is_forward  total_weight_all_edges  forward_weight_begin  forward_weight_end  permanent_estimate  removed_edges_count  removed_weight
720575940629970489 720575940631267655     8.0   0.0    0.0        False        True              41912141.0            35462925.0          35462925.0                 0.0                   50           221.0
720575940619127256 720575940610718328     3.0   0.0    0.0         True       False              41912141.0            35462925.0          35462925.0                 0.0                   50           221.0
720575940619127256 720575940609685717     2.0   0.0    0.0         True       False              41912141.0            35462925.0          35462925.0                 0.0                   50           221.0
720575940619127256 720575940628662528     2.0   0.0    0.0         True       False              41912141.0            35462925.0          35462925.